# HW 3: Continuous Heterogeneity

Re-estimate the multinomial logit brand choice model from last week using a continuous heterogeneity distribution and the simulated maximum likelihood approach discussed today.

Please use the attached set of simulated normal draws for your random coefficients:
- M=100 and since N=100, there are 10k draws. 
- Use a unique set of 100 for each individual
- Each m draw is a 1 by 7 vector to represent the 4 intercepts and 3 slope coefficients

In [1]:
# Packages
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import norm
from statsmodels.tools.numdiff import approx_hess
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed
import time

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from jax.scipy.special import logsumexp as jax_logsumexp

In [2]:
# Import data
N = 100
S = 100

eta_raw = np.loadtxt("../data/eta.txt")
eta = eta_raw.reshape(N, S, 7, order="C")  # first 100 rows are for the first household, etc.
df = pd.read_csv("../data/YogurtLong.txt", sep="\t")
df.head()

,hh,qty,exp,nopurch,b1,b2,b3,b4,p1,p2,p3,p4,f1,f2,f3,f4,tripnum
0,1,2,40.900002,0,0,0,0,1,0.11,0.08,0.06,0.08,0,0,0,0,1
1,1,0,8.830000,1,0,0,0,0,0.11,0.09,0.05,0.08,0,0,0,0,2
2,1,0,3.880000,1,0,0,0,0,0.13,0.09,0.05,0.08,0,0,0,0,3
3,1,0,0.780000,1,0,0,0,0,0.13,0.09,0.05,0.08,0,0,0,0,4
4,1,0,39.240002,1,0,0,0,0,0.13,0.09,0.05,0.08,0,0,0,0,5


In [3]:
# Convert df to long (each row is a hh-trip-brand choice occasion)
df_long = (
    pd.wide_to_long(
        df,
        stubnames=["b", "p", "f"],
        i=["hh", "tripnum"],
        j="brand",
        suffix=r"\d+"
    )
    .reset_index()
    .sort_values(["hh", "tripnum", "brand"])
)
df_long = pd.get_dummies(df_long, columns=["brand"], prefix="brand", dtype=int)

# Create prev_purch variable
df_long["brand_num"] = df_long[["brand_1", "brand_2", "brand_3", "brand_4"]].idxmax(axis=1).str.replace("brand_", "").astype(int)
trip_choice = (
    df_long.loc[df_long["b"] == 1, ["hh", "tripnum", "brand_num"]]
    .rename(columns={"brand_num": "chosen_brand"})
)
trip_hist = (
    df_long[["hh", "tripnum"]]
    .drop_duplicates()
    .sort_values(["hh", "tripnum"])
    .merge(trip_choice, on=["hh", "tripnum"], how="left")
)
trip_hist["prev_purch"] = (
    trip_hist.groupby("hh")["chosen_brand"]
    .transform(lambda x: x.ffill().shift(1))
    .fillna(0)
    .astype(int)
)
df_long = df_long.merge(
    trip_hist[["hh", "tripnum", "prev_purch"]],
    on=["hh", "tripnum"],
    how="left",
)
df_long["prev_purch"] = (df_long["prev_purch"] == df_long["brand_num"]).astype(int)

df_long.head()

,hh,tripnum,exp,nopurch,qty,b,p,f,brand_1,brand_2,brand_3,brand_4,brand_num,prev_purch
0,1,1,40.900002,0,2,0,0.11,0,1,0,0,0,1,0
1,1,1,40.900002,0,2,0,0.08,0,0,1,0,0,2,0
2,1,1,40.900002,0,2,0,0.06,0,0,0,1,0,3,0
3,1,1,40.900002,0,2,1,0.08,0,0,0,0,1,4,0
4,1,2,8.830000,1,0,0,0.11,0,1,0,0,0,1,0


## Random coefficients implementation

Suppose there are $N$ households $i=1, \ldots, N$, each of whom makes $T_i$ trips and decides between $J$ products or an outside option. For simplicity, we drop the subscript on $T_i$. In total, there are $NT$ choice occasions.

In the random coefficient model, the set of coefficients are household-specific and distributed according to a continuous parametric distribution $g(\bm{\theta})$. For example, $g$ could be the normal distribution parameterized by $\bm{\theta} = (\bm{\bar\beta}, \bm\Sigma)$:
$$
\bm{\beta}_i 
\coloneqq \begin{bmatrix} \beta_{1i} \\ \beta_{2i} \\ \vdots \\ \beta_{ki} \end{bmatrix} 
\sim \mathcal{N}\left( 
    \begin{bmatrix} 
    \bar\beta_{1} \\ \bar\beta_{2} \\ \vdots \\ \bar\beta_{k} 
    \end{bmatrix}, 
    \begin{bmatrix} 
    \sigma_1^2 & \sigma_{1,2} & \cdots & \sigma_{1,k} \\
    \sigma_{2,1} & \sigma_2^2 & \cdots & \sigma_{2,k} \\
    \vdots & \vdots & \ddots & \vdots \\
    \sigma_{k, 1} & \sigma_{k, 2} & \cdots & \sigma_k^2
    \end{bmatrix} 
    \right)
\eqqcolon \mathcal{N}(\bm{\bar\beta}, \bm\Sigma)
$$
Equivalently, letting $\bm{\Gamma} \in \mathbb{R}^{k \times k}$ be the Cholesky decomposition of $\bm\Sigma$ (i.e., $\bm\Sigma = \bm{\Gamma}\bm{\Gamma}^\intercal$),
$$
\bm{\beta}_{i} = \bm{\bar\beta} + \bm\Gamma \eta_i, \quad \eta_i \sim \mathcal{N}(0, I_{k \times k})
$$
Note that $\bm{\Gamma}$ is lower-triangular, so there are only $\frac{k(k+1)}{2}$ parameters to estimate to identify the covariance matrix. If there is no perfect multicollinearity among the brand intercepts, then $\bm{\Sigma}$ is positive definite and the Cholesky factorization $\bm{\Gamma}$ is unique.

The likelihood of household $i$'s observed choices $\bm{y}_i = (y_{i1}, \ldots, y_{iT})$ given their covariates $\bm{x}_i = (x_{i1}, \ldots, x_{iT})$ and the distribution parameters $\bm\theta$ is
$$\begin{align*}
L_i 
&= L(\bm{y}_i \mid \bm{x}_i, \bm\theta) \\
&= \int \prod_{t=1}^T L(y_{it} \mid x_{it}, \bm\beta) g(\bm\beta \mid \bm\theta) \mathrm{d} \bm\beta \\
&= \int \prod_{t=1}^T L(y_{it} \mid x_{it}, \bar{\bm\beta}, \bm{\Gamma}, \eta) \phi(\eta) \mathrm{d} \eta
\end{align*}$$
where $\phi(\eta)$ is the pdf of the standard normal. To simulate the integral, we take $S$ draws of $\eta$ according to $\phi(\eta)$. Call these draws $\eta^{(1)}, \ldots, \eta^{(S)}$. The likelihood can be approximated as
$$
\tilde{L}_i 
= \frac{1}{S} \sum_{s=1}^S L_i(\bm{y}_i \mid \bm{x}_i, \bar{\bm\beta}, \bm{\Gamma}, \eta^{(s)})
= \frac{1}{S} \sum_{s=1}^S \prod_{t=1}^T L(y_{it} \mid x_{it}, \bar{\bm\beta}, \bm{\Gamma}, \eta^{(s)})
$$
To compute this likelihood, we use a similar method as described in `2-discrete-heterogeneity.ipynb`. At a high level, the key difference is that we replace classes with the $S$ draws. Let $\bm{\Eta}_N \in \mathrm{R}^{S \times k \times N}$ be a matrix of random draws from the standard normal and let $\bar{\bm{B}} \in \mathbb{R}^{S \times k \times N}$ be $\bar{\bm\beta}$ copied $S \times N$ times in the conforming directions (i.e. $\bar{\bm\beta}[\texttt{None}\;,:,\;\texttt{None}]$ in Python). Then, the simulated coefficients for all households for all draws are contained in
$$
\bm{B} = \bar{\bm{B}} + \bm{\Eta}_N \bm{\Gamma}^\intercal \in \mathbb{R}^{S \times k \times N}
$$
Given the design matrix $X \in \mathbb{R}^{NTJ \times k}$

For numerical stability, it is better to work in log space. Let $j_{it}^* \in \{0, \ldots, J\}$ be the product chosen by household $i$ in trip $t$. Then,



$$\begin{align*}
\log(\tilde{L}_i) 
&= \log\left(\frac{1}{S} \sum_{s=1}^S \prod_{t=1}^T L(y_{it} \mid x_{it}, \bar{\bm\beta}_0, \bm{\Gamma}, \bm{\beta}_{\mathrm{com}}, \eta^{(s)}\right) \\
&= -\log(S) + \log\left(\sum_{s=1}^S \exp\left( \log\left( \prod_{t=1}^T L(y_{it} \mid x_{it}, \bar{\bm\beta}_0, \bm{\Gamma}, \bm{\beta}_{\mathrm{com}}, \eta^{(s)}\right)\right)\right) \\
&= -\log(S) + \log\left(\sum_{s=1}^S \exp\left( \log\left( \prod_{t=1}^T \frac{\exp(x_{itj}\bm\beta_{j_{it}^*}^{(s)})}{1+\sum_{j=1}^J \exp(x_{itj}\bm\beta_{j}^{(s)}} \right)\right)\right) \\
&= -\log(S) + \mathrm{logsumexp}_s \left( \sum_{t=1}^T \left( x_{itj}\bm\beta_{j_{it}^*}^{(s)} - \mathrm{logsumexp}_j ([0, x_{itj}\bm\beta_{j}^{(s)}] ) \right)\right) \\
\end{align*}$$
where
$$
\bm\beta^{(s)} = \begin{bmatrix} \bar{\bm\beta}_0 + \bm{\Gamma}\eta^{(s)} \\ \bm{\beta}_{\mathrm{com}} \end{bmatrix}, \quad \bm\beta_{j}^{(s)}
$$

### Aside: Allowing for common coefficients via restrictions on covariance matrix

Suppose we don't want all $k$ coefficients to be household-specific. For example, we may only want brand coefficients $\bm{\bm\beta}_{0i}$ to be heterogeneous while other coefficients $\beta_1, \ldots, \beta_{\tilde{k}}$ are shared by all households. In this case, we can express the distribution of the covariate vector as
$$
\bm{\beta}_i = \begin{bmatrix} \bm{\bm\beta}_{0i} \\ \beta_1 \\ \vdots \\ \beta_{\tilde{k}} \end{bmatrix}
\sim \mathcal{N}\left( 
    \begin{bmatrix} 
    \bm{\bar\beta_0} \\ \beta_1 \\ \vdots \\ \beta_{\tilde{k}} 
    \end{bmatrix}, 
    \begin{bmatrix} 
    \bm\Sigma & \bm{0}_{J \times \tilde{k}} \\
    \bm{0}_{\tilde{k} \times J} & \bm{0}_{\tilde{k} \times \tilde{k}}
    \end{bmatrix} 
    \right)
\eqqcolon \mathcal{N}(\bm{\bar\beta}, \tilde{\bm\Sigma})
$$
That is, we restrict variances and covariances corresponding to common coefficients to be 0. The corresponding Cholesky decomposition form is
$$
\bm{\beta}_i = \bm{\bar\beta} + \bm\Gamma \eta_i, \quad \eta_i \sim \mathcal{N}(0, I_{J \times J}) 
$$
where
$$
\bm\Gamma = 
\begin{bmatrix} 
    \tilde{\bm\Gamma} & \bm{0}_{J \times \tilde{k}} \\
    \bm{0}_{\tilde{k} \times J} & \bm{0}_{\tilde{k} \times \tilde{k}}
\end{bmatrix}, \quad \tilde{\bm\Gamma}\tilde{\bm\Gamma}^\intercal = \bm\Sigma
$$
The parameters to be estimated are the $k$ coefficients in $\bm{\bar\beta}$ and the $\frac{J(J+1)}{2}$ parameters in $\tilde{\bm{\Gamma}}$.

In [4]:
def _panel_indices(hh_ids, n_alts, n_rows):
    """Map each row of the long panel to a household index in 0..N-1."""
    hh_ids = np.asarray(hh_ids).reshape(-1)
    n_choices = n_rows // n_alts
    hh_trip = hh_ids.reshape(n_choices, n_alts, order="C")[:, 0]
    _, hh_index = np.unique(hh_trip, return_inverse=True)
    return hh_index


def _unpack_theta(theta, k, k_het):
    """
    Unpack theta into mean coefficients (k,) and Cholesky factor for heterogeneous coefficients.

    Theta ordering is [mean coefficients (length k), lower-triangular elements of Γ̃ (length k_het(k_het+1)/2)].

    Γ is k × k_het with Γ̃ in the top k_het rows (lower triangular) and zeros below, so that
    β = β̄ + Γ η with η ~ N(0, I_{k_het}) and common coefficients have zero variance.
    Columns of X must be ordered with heterogeneous covariates first, then common (see ``estimate_MNL``).
    """
    p = k_het * (k_het + 1) // 2
    betas = theta[:k]
    gamma_flat = theta[k : k + p]
    Gamma_tilde = jnp.zeros((k_het, k_het))
    tril_i, tril_j = jnp.tril_indices(k_het)
    Gamma_tilde = Gamma_tilde.at[tril_i, tril_j].set(gamma_flat)
    if k_het == k:
        Gamma = Gamma_tilde
    else:
        Gamma = jnp.concatenate(
            [Gamma_tilde, jnp.zeros((k - k_het, k_het))],
            axis=0,
        )
    return betas, Gamma


def _negloglik_hh(theta, X, Y, eta, hh_index, n_alts, k_het, N=None):
    """
    Negative log-likelihoods for random coefficient MNL with outside option.
    Returns (N,) — one value per household.
    """
    N = int(np.max(np.asarray(hh_index))) + 1
    k = X.shape[1]
    n_choices = Y.shape[0] // n_alts

    theta = jnp.asarray(theta, dtype=jnp.float64).reshape(-1)
    X = jnp.asarray(X, dtype=jnp.float64)
    Y = jnp.asarray(Y, dtype=jnp.float64).reshape(-1)
    eta = jnp.asarray(eta, dtype=jnp.float64)
    hh_index = jnp.asarray(hh_index, dtype=jnp.int32)

    n_draws = eta.shape[1]                                  # S
    mean_beta, Gamma = _unpack_theta(theta, k, k_het)      # (k,) and (k, k_het)
    B = mean_beta[None, None, :] + eta @ Gamma.T            # (N, S, k)
    hh_index1 = jnp.repeat(hh_index, n_alts)
    B_hh = B[hh_index1, :, :]                               # (NTJ, S, k)

    V = jnp.einsum("ij,isj->is", X, B_hh)                   # (NTJ, S)
    V = V.reshape(n_choices, n_alts, n_draws, order="C")    # (NT, J, S)
    V0 = jnp.concatenate([jnp.zeros((n_choices, 1, n_draws)), V], axis=1)
    Y1 = Y.reshape(n_choices, n_alts, 1, order="C")         # (NT, J, 1)
    chosen_v = jnp.multiply(Y1, V).sum(axis=1)              # (NT, S)
    log_L3 = chosen_v - jax_logsumexp(V0, axis=1)           # (NT, S)

    log_L2 = jax.ops.segment_sum(log_L3, hh_index, num_segments=N)  # (N_hh, S)
    log_L1 = -jnp.log(jnp.asarray(n_draws, dtype=theta.dtype)) + jax_logsumexp(log_L2, axis=1)  # (N,)   
    return -log_L1


def _negloglik(theta, X, Y, eta, hh_index, n_alts, k_het):
    """Summed negative log-likelihood."""
    negLL = _negloglik_hh(theta, X, Y, eta, hh_index, n_alts, k_het)
    return np.sum(negLL)


def _score_matrix(theta, X, Y, eta, hh_index, n_alts, k_het):
    """
    Score matrix of household-level negative log-likelihood: d (NLL_h) / d theta.
    Shape (n_households, n_params). Computed with JAX reverse-mode Jacobian.
    """
    def neglog_hh(th):
        return _negloglik_hh(th, X, Y, eta, hh_index, n_alts, k_het)

    J = jax.jacrev(neglog_hh)(theta)
    return np.asarray(J)


def _print_mle_output(output, df, n_alts, covariate_vars, robust_se, individual_var, k_het):
    """Prints MLE results in a nice table."""
    
    B_hat = np.asarray(output['opt_beta'], dtype=float).reshape(-1)
    Sigma_hat = np.asarray(output['opt_sigma'])
    
    n_choices = df.shape[0] // n_alts               # NT
    n_households = df[individual_var].nunique()     # N
    k = B_hat.shape[0]

    se_theta = np.asarray(output['se_theta'], dtype=float).reshape(-1)
    ci_theta = np.asarray(output['ci_theta'], dtype=float)
    if ci_theta.ndim == 1:
        ci_theta = ci_theta.reshape(-1, 2)

    se_beta = np.zeros(k, dtype=float)
    ci_beta = np.zeros((k, 2), dtype=float)

    se_beta[:k] = se_theta[:k]
    ci_beta[:k, :] = ci_theta[:k, :]

    print(output.get('message', ''))
    print(f"Converged in {output['num_iter']} iterations.")
    print()
    print(f"Log-likelihood: {output['opt_ll']:.4f}")
    print(f"Number of individuals: {n_households}")
    print(f"Number of choice occasions: {n_choices}")
    if robust_se:
        print("Robust standard errors, clustered on the", individual_var, "variable.")
    else:
        print("Standard errors are not robust and assume correct model specification.")
    print()

    widths = (12, 10, 11, 10, 10)
    header_top = f"{'Coefficient':^{widths[0]}} | {'Estimate':^{widths[1]}} | {'Std. Err.':^{widths[2]}} | {'[Confidence Interval]':^{widths[3] + widths[4] + 3}}"
    divider = "-+-".join("-" * w for w in widths)

    print("Population Mean of Coefficients")
    print(header_top)
    print(divider)
    for i, name in enumerate(covariate_vars):
        row = " | ".join([
            f"{name:<{widths[0]}}",
            f"{B_hat[i]:>{widths[1]}.6f}",
            f"{se_beta[i]:>{widths[2]}.6f}",
            f"{ci_beta[i, 0]:>{widths[3]}.5f}",
            f"{ci_beta[i, 1]:>{widths[4]}.5f}"
        ])
        print(row)
    print()

    print("Covariance matrix")
    cov_print = pd.DataFrame(
        np.round(Sigma_hat[:k_het, :k_het], 4), 
        index=covariate_vars[:k_het], 
        columns=covariate_vars[:k_het])
    upper = np.triu(np.ones(cov_print.shape, dtype=bool), k=1)
    print(cov_print.mask(upper).to_string(na_rep=""))

def _mle_estimator(df,
                   choice_var,
                   covariate_vars,
                   hh_index,
                   n_alts,
                   eta,
                   beta_init,
                   gamma_init,
                   k_het,
                   ci_alpha,
                   robust_se,
                   opt_method):
    """
    Estimates the multinomial logit model with random coefficients using maximum likelihood estimation,
    from initial values (beta_init, gamma_init).

    k_het : int
        Number of heterogeneous coefficients (columns of X that load on η). Must match eta.shape[2].
    """
    # Prepare data
    Y = df[[choice_var]].to_numpy()
    X = df[covariate_vars].to_numpy()
    k = X.shape[1]

    # Analytic gradient (JAX reverse-mode)
    _X_j = jnp.asarray(X, dtype=jnp.float64)
    _Y_j = jnp.asarray(Y, dtype=jnp.float64).reshape(-1)
    _eta_j = jnp.asarray(eta, dtype=jnp.float64)
    _hh_j = jnp.asarray(hh_index, dtype=jnp.int32)

    def _nll_sum_jax(th):
        return jnp.sum(
            _negloglik_hh(th, _X_j, _Y_j, _eta_j, _hh_j, n_alts, k_het)
        )

    # JIT-compiled gradient of neg LL w.r.t. theta
    _grad_nll_jit = jax.jit(jax.grad(_nll_sum_jax))
    
    def _jac_negloglik(theta, *args):
        g = _grad_nll_jit(jnp.asarray(theta, dtype=jnp.float64).reshape(-1))
        return np.asarray(g, dtype=float)

    theta_init = np.concatenate([beta_init, gamma_init])

    # Run optimization
    result = minimize(
        fun=_negloglik,
        jac=_jac_negloglik,
        x0=theta_init,
        args=(X, Y, eta, hh_index, n_alts, k_het),
        method=opt_method
    )

    # Extract results
    opt_ll = -result.fun
    opt_theta = result.x
    B_hat, Gamma_hat = _unpack_theta(opt_theta, k, k_het)
    Sigma_hat = Gamma_hat @ Gamma_hat.T

    # Compute standard errors and confidence intervals using the Hessian
    H = approx_hess(opt_theta, _negloglik, args=(X, Y, eta, hh_index, n_alts, k_het))
    H_inv = np.linalg.pinv(H)
    se_theta = np.sqrt(np.clip(np.diag(H_inv), 0.0, None))  # avoid negative variances from numerical issues

    z_score = norm.ppf(1 - ci_alpha / 2)
    ci_theta = [(coef - z_score * se, coef + z_score * se) for coef, se in zip(opt_theta, se_theta)]

    # Collect output
    output = {
        'opt_ll': opt_ll,
        'opt_theta': opt_theta,
        'opt_beta': B_hat,
        'opt_gamma': Gamma_hat,
        'opt_sigma': Sigma_hat,
        'se_theta': se_theta,
        'ci_theta': ci_theta,
        'success': result.success,
        'message': result.message,
        'num_iter': result.nit,
    }

    if robust_se:
        # J estimated by outer products of household-level scores.
        S = _score_matrix(opt_theta, X, Y, eta, hh_index, n_alts, k_het)
        J = S.T @ S

        # Sandwich variance: H^{-1} J H^{-1}
        V_robust = H_inv @ J @ H_inv
        se_theta_r = np.sqrt(np.clip(np.diag(V_robust), 0.0, None))
        ci_r = [(coef - z_score * se, coef + z_score * se) for coef, se in zip(opt_theta, se_theta_r)]

        output['se_theta'] = se_theta_r
        output['ci_theta'] = ci_r
        output['param_cov'] = V_robust
    else:
        output['param_cov'] = H_inv

    return output


def _run_mle_from_start(start_idx, beta0, gamma0, estimator_kwargs):
    """Run one MLE start and return its index, output, and elapsed time."""
    t_start = time.time()
    out = _mle_estimator(beta_init=beta0, gamma_init=gamma0, **estimator_kwargs)
    t_end = time.time()
    out['elapsed_time'] = t_end - t_start
    return start_idx, out


def estimate_MNL(df,
                 choice_var,
                 covariate_vars,
                 individual_var,
                 n_alts,
                 eta,
                 common_vars=None,
                 beta_init=None,
                 gamma_init=None,
                 ci_alpha=0.05,
                 robust_se=False,
                 randomized_starts=0,
                 search_bounds=[-10, 10],
                 seed=None,
                 n_cores=1,
                 opt_method='BFGS',
                 output_log=False):
    """
    Estimates the multinomial logit model with random coefficients using maximum likelihood estimation.
    
    Optional randomized starts are drawn uniformly from the specified `search_bounds` centered around 
    (`beta_init`, `gamma_init`) to help avoid local optima. Specifying `n_cores` > 1 will evaluate these 
    starts in parallel using multiprocessing.
    
    Parameters    
    ----------
    df : DataFrame
        The dataset to be used for estimation.
    choice_var : str
        The column name of the choice variable.
    covariate_vars : list
        Column names in estimation order: list every heterogeneous covariate first, then every common
        covariate (fixed coefficient). When ``common_vars`` is non-empty, its entries must match the
        last ``len(common_vars)`` names here, in the same order.
    individual_var : str
        The column name of the individual ID variable.
        If robust standard errors are requested, standard errors are clustered by this variable.
    n_alts : int
        Number of alternatives.
    eta : array
        Random draws from the standard normal. Shape (N, S, k) with columns aligned to
        ``covariate_vars``. Only the first k_het columns (heterogeneous block) are used in simulation.
    common_vars : list or None
        Names of covariates whose coefficients are fixed across households (zero rows/columns in Σ).
        If None (default), all covariates are random. If provided, must equal
        ``covariate_vars[-len(common_vars):]`` (heterogeneous names come first in ``covariate_vars``).
    beta_init : array, optional
        Array of shape (k,) with initial guess for coefficients. 
        If None, defaults to zeros.
    gamma_init : array, optional
        Array of shape (k_het, k_het) with initial guess for Cholesky factor of covariance matrix, 
        where k_het is the number of heterogeneous covariates.
        If None, defaults to identity matrix.
    ci_alpha : float
        Significance level for confidence intervals.
    robust_se : bool
        Whether to compute robust standard errors. 
        If True, standard errors are clustered by `individual_var`.
    randomized_starts : int
        Number of additional random starting points to use for optimization.
    search_bounds : list
        Bounds for random perturbations around initial values, specified as [lower_bound, upper_bound].
    seed : int or None
        Random seed for reproducibility of randomized starts.
    n_cores : int
        Number of worker processes (CPU cores) for evaluating randomized starts in parallel.
        If 1 (default), evaluates starts serially.
        Use an integer > 1 to opt in to parallel processing.
    opt_method : str
        Optimization method.
    output_log : bool
        If True, include a 'start_log' DataFrame in output showing achieved LL, elapsed time, 
        optimization status and message, starting params, parameter estimates for each start.

    Returns
    -------
    dict
        Includes ``opt_ll``, ``opt_theta``, ``opt_beta``, ``opt_gamma``, ``opt_sigma``, ``se_theta``,
        ``ci_theta``, ``param_cov`` ((k+p)×(k+p) covariance used for SEs), and optionally ``start_log``.
    """
    k = len(covariate_vars)
    common_vars = [] if common_vars is None else list(common_vars)
    unknown = set(common_vars) - set(covariate_vars)
    if unknown:
        raise ValueError(f"common_vars contains names not in covariate_vars: {unknown}")
    k_common = len(common_vars)
    k_het = k - k_common
    if k_het < 1:
        raise ValueError("Need at least one heterogeneous coefficient (check common_vars).")
    if k_common > 0 and covariate_vars[-k_common:] != common_vars:
        raise ValueError(
            "covariate_vars must list heterogeneous covariates first, then common covariates. "
            "Require covariate_vars[-len(common_vars):] == common_vars (same order)."
        )
    p = k_het * (k_het + 1) // 2

    eta = np.asarray(eta)
    n_hh = df[individual_var].nunique()
    if eta.shape[0] != n_hh or eta.shape[2] != k:
        raise ValueError(f"eta must have shape (N, S, k)=({n_hh}, S, {k}); got {eta.shape}")
    eta_het = eta[:, :, :k_het]

    individual_ids = df[individual_var].to_numpy()
    hh_index = _panel_indices(individual_ids, n_alts, df.shape[0])

    if beta_init is None:
        beta_base = np.zeros(k)
    else:
        beta_base = np.asarray(beta_init).reshape(-1)

    if gamma_init is None:
        gamma_mat = np.eye(k_het)
    else:
        gamma_mat = np.asarray(gamma_init)
        if gamma_mat.shape != (k_het, k_het):
            raise ValueError(f"gamma_init must have shape ({k_het}, {k_het}); got {gamma_mat.shape}")
    gamma_base = gamma_mat[np.tril_indices(k_het)].reshape(-1)

    # Generate randomized starts
    rng = np.random.default_rng(seed)
    n_rand = max(int(randomized_starts)-1, 0)
    beta_perturbs = rng.uniform(low=search_bounds[0], high=search_bounds[1], size=(n_rand, k))
    beta_rand = beta_base[None, :] + beta_perturbs          # (n_rand, k)

    gamma_perturbs = rng.uniform(low=search_bounds[0], high=search_bounds[1], size=(n_rand, p))
    gamma_rand = gamma_base[None, :] + gamma_perturbs       # (n_rand, p)
    
    starts = [(beta_base.copy(), gamma_base.copy())] + list(zip(beta_rand, gamma_rand))

    # Assign starts to workers and run optimizations
    estimator_kwargs = {
        'df': df,
        'choice_var': choice_var,
        'covariate_vars': covariate_vars,
        'hh_index': hh_index,
        'n_alts': n_alts,
        'eta': eta_het,
        'k_het': k_het,
        'ci_alpha': ci_alpha,
        'robust_se': robust_se,
        'opt_method': opt_method,
    }

    n_starts = len(starts)
    max_workers = max(1, int(n_cores))
    max_workers = min(max_workers, n_starts)

    best_output = None
    start_log = []
    start_outputs = {}

    if max_workers > 1:
        try:
            print(f"Evaluating {n_starts} randomized starts in parallel using {max_workers} cores...")
            mp_context = multiprocessing.get_context('fork')
            with ProcessPoolExecutor(max_workers=max_workers, mp_context=mp_context) as ex:
                futures = [
                    ex.submit(_run_mle_from_start, i, beta0, gamma0, estimator_kwargs)
                    for i, (beta0, gamma0) in enumerate(starts)
                ]
                for fut in as_completed(futures):
                    start_idx, out = fut.result()
                    if output_log:
                        start_log.append((start_idx, out['opt_ll'], out['elapsed_time'], out['success'], 
                                          out['message'], starts[start_idx][0], starts[start_idx][1],
                                          out['opt_beta'], out['opt_gamma'], out['opt_sigma']))
                        start_outputs[start_idx] = out
                    if (best_output is None) or (out['opt_ll'] > best_output['opt_ll']):
                        best_output = out
        except Exception as exc:
            print(f'Parallel starts failed ({exc}); falling back to serial evaluation.')
            for i, (beta0, gamma0) in enumerate(starts):
                _, out = _run_mle_from_start(i, beta0, gamma0, estimator_kwargs)
                if output_log:
                    start_log.append((i, out['opt_ll'], out['elapsed_time'], out['success'], 
                                      out['message'], starts[i][0], starts[i][1],
                                      out['opt_beta'], out['opt_gamma'], out['opt_sigma']))
                    start_outputs[i] = out
                if (best_output is None) or (out['opt_ll'] > best_output['opt_ll']):
                    best_output = out
    else:
        for i, (beta0, gamma0) in enumerate(starts):
            _, out = _run_mle_from_start(i, beta0, gamma0, estimator_kwargs)
            if output_log:
                start_log.append((i, out['opt_ll'], out['elapsed_time'], out['success'], 
                                  out['message'], starts[i][0], starts[i][1],
                                  out['opt_beta'], out['opt_gamma'], out['opt_sigma']))
                start_outputs[i] = out
            if (best_output is None) or (out['opt_ll'] > best_output['opt_ll']):
                best_output = out

    _print_mle_output(output=best_output, df=df, n_alts=n_alts,
                      covariate_vars=covariate_vars, robust_se=robust_se, 
                      individual_var=individual_var, k_het=k_het)
    
    if output_log:
        log_df = pd.DataFrame(start_log, columns=[
            'start_idx', 'achieved_ll', 'elapsed_time', 'success', 'message', 
            'start_beta', 'start_gamma', 'opt_beta', 'opt_gamma', 'opt_sigma'])
        log_df.sort_values('achieved_ll', ascending=False, inplace=True)
        best_output['start_log'] = log_df.reset_index(drop=True)
        best_output['start_outputs'] = start_outputs
    
    return best_output

### Full heterogeneity

In [5]:
beta_init = [0,0,0,0,0,0,0]
gamma_init = np.eye(7)

results_fullhet = estimate_MNL(
    df=df_long,
    choice_var='b',
    covariate_vars=['brand_1', 'brand_2', 'brand_3', 'brand_4', 'f', 'p', 'prev_purch'],
    individual_var='hh',
    n_alts=4,
    eta=eta,
    common_vars=None,
    beta_init=beta_init,
    gamma_init=gamma_init,
    robust_se=True,
    randomized_starts=1,
    search_bounds=[-5, 5],
    n_cores=1,
    opt_method='BFGS',
    output_log=True
)

Optimization terminated successfully.
Converged in 123 iterations.

Log-likelihood: -6365.1148
Number of individuals: 100
Number of choice occasions: 12784
Robust standard errors, clustered on the hh variable.

Population Mean of Coefficients
Coefficient  |  Estimate  |  Std. Err.  |  [Confidence Interval] 
-------------+------------+-------------+------------+-----------
brand_1      |   0.251410 |    0.285720 |   -0.30859 |    0.81141
brand_2      |  -0.053385 |    0.312690 |   -0.66625 |    0.55948
brand_3      |  -3.545602 |    0.272885 |   -4.08045 |   -3.01076
brand_4      |  -1.040263 |    0.284430 |   -1.59774 |   -0.48279
f            |   0.491617 |    0.168580 |    0.16121 |    0.82203
p            | -39.878073 |    2.816609 |  -45.39853 |  -34.35762
prev_purch   |   1.853407 |    0.268773 |    1.32662 |    2.38019

Covariance matrix
            brand_1  brand_2  brand_3  brand_4       f         p  prev_purch
brand_1      5.8068                                                

### Heterogeneity only on brand intercepts

Put heterogeneous covariates first in `covariate_vars`, then common ones; `common_vars` must match the trailing names in that order (here: four brand dummies, then `f`, `p`, `prev_purch`). Brand intercepts are random; `f`, `p`, and `prev_purch` are fixed (zeros in the corresponding rows/columns of Σ). Initial `gamma_init` is `k_het × k_het` (here 4×4).

In [6]:
beta_init = [0.5, -0.5, -3, -2, 0.5, -35, 1.5]
gamma_init = np.eye(4) * 3

results_brandhet = estimate_MNL(
    df=df_long,
    choice_var="b",
    covariate_vars=["brand_1", "brand_2", "brand_3", "brand_4", "f", "p", "prev_purch"],
    individual_var="hh",
    n_alts=4,
    eta=eta,
    common_vars=["f", "p", "prev_purch"],
    beta_init=beta_init,
    gamma_init=gamma_init,
    robust_se=True,
    randomized_starts=1,
    search_bounds=[-5, 5],
    n_cores=1,
    opt_method="BFGS",
    output_log=True,
)

Optimization terminated successfully.
Converged in 54 iterations.

Log-likelihood: -6455.0933
Number of individuals: 100
Number of choice occasions: 12784
Robust standard errors, clustered on the hh variable.

Population Mean of Coefficients
Coefficient  |  Estimate  |  Std. Err.  |  [Confidence Interval] 
-------------+------------+-------------+------------+-----------
brand_1      |   0.273658 |    0.408009 |   -0.52602 |    1.07334
brand_2      |  -0.297920 |    0.314706 |   -0.91473 |    0.31889
brand_3      |  -3.973147 |    0.388733 |   -4.73505 |   -3.21124
brand_4      |  -2.466908 |    0.381048 |   -3.21375 |   -1.72007
f            |   0.375376 |    0.136073 |    0.10868 |    0.64207
p            | -39.209148 |    3.713217 |  -46.48692 |  -31.93138
prev_purch   |   1.299516 |    0.125860 |    1.05283 |    1.54620

Covariance matrix
         brand_1  brand_2  brand_3  brand_4
brand_1   0.9294                           
brand_2  -0.1168   2.3685                  
brand_3  -0.2